# 02_augmentation.ipynb - Augmentation and Combined Dataset Creation

**Project**: BSM Project 3 - D-ResNet Paper Reproduction
**Paper**: "D-Resnet: deep residual neural network for exploration, identification, and classification of beach sand minerals" (Multimedia Tools and Applications, 2024)
**Step**: 2 of 9 - Augmentation

## Objectives
1. Load original dataset (367 images, 6 classes)
2. Implement paper's augmentation: horizontal flip + rotations (0Â°, 30Â°, 45Â°, 90Â°, 180Â°, 270Â°)
3. Add pixel translation (Â±10-15% shift) as extension
4. Resize all to 224Ã—224Ã—3, normalize with ImageNet mean/std
5. Generate augmented images so total combined dataset â‰ˆ 1468 (4x), roughly balanced per class
6. Save combined dataset for reuse in all subsequent notebooks
7. Fixed random seed (42) everywhere

In [ ]:
# ========== CONFIGURATION ==========
SEED = 42
DATA_DIR = "/kaggle/input/beach-sand-mineral-bsm/BSM"  # Kaggle mount path# Alternative local path for testing:
# DATA_DIR = "C:/BSM Project 3/data_raw/BSM"

EXPECTED_CLASSES = ["Garnet", "Ilmenite", "Monazite", "Rutile", "Sillimanite", "Zircon"]  # Actual folder names
CLASS_NAMES = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]  # Paper order

# Paper augmentation: horizontal flip + 6 rotations = 2 * 6 = 12 variants per image
# But paper says 1468 total = 367 * 4, so they used subset of rotations
# Let's calculate: 367 * 4 = 1468, so 4 variants per image (including original)
# Paper rotations: 0, 30, 45, 90, 180, 270 (6 rotations) + horizontal flip
# If they use all: 367 * 12 = 4404 (too many)
# If they use 3 rotations + flip: 367 * 6 = 2202 (still too many)
# If they use 3 rotations (no flip): 367 * 3 = 1101 (too few)
# Likely: Original + 3 augmented = 4x, using subset of rotations

TARGET_TOTAL = 1468  # Paper's augmented total
TARGET_PER_CLASS = TARGET_TOTAL // 6  # ~244 per class

IMG_SIZE = (224, 224)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Output paths
OUTPUT_DIR = "/kaggle/working/combined_dataset"
RESULTS_DIR = "/kaggle/working/results"
DATA_INFO_DIR = "/kaggle/working/data_info"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_INFO_DIR, exist_ok=True)

In [ ]:
# ========== IMPORTS & SEED ==========
import os
import sys
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import hashlib
import shutil

import cv2
from PIL import Image, ImageOps
import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)

# Albumentations seed
A.seed(SEED)

# Print versions
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"PIL: {Image.__version__}")
print(f"Albumentations: {A.__version__}")
print(f"Seed: {SEED}")

In [ ]:
# ========== LOAD ORIGINAL DATASET ==========
# Try multiple possible mount paths
from pathlib import Path

possible_paths = [
    Path("/kaggle/input/beach-sand-mineral-bsm/BSM"),
    Path("/kaggle/input/beach-sand-mineral-bsm"),
    Path("/kaggle/input/prasannavenkatesant/beach-sand-mineral-bsm/BSM"),
    Path("/kaggle/input/prasannavenkatesant/beach-sand-mineral-bsm"),
    Path("/kaggle/input") / DATASET_SLUG.replace("/", "-") / "BSM",
    Path("/kaggle/input") / DATASET_SLUG.replace("/", "-"),
    Path("./data/BSM"),
    Path("./data"),
]

data_path = None
for p in possible_paths:
    if p.exists():
        data_path = p
        print(f"Found data at: {data_path}")
        break

if data_path is None:
    print("ERROR: Could not find dataset at any expected location!")
    print("Please check the dataset is attached to this notebook.")
    # Use default for local testing
    data_path = Path(DATA_DIR)

print(f"Data path: {data_path}")
print(f"Exists: {data_path.exists()}")

In [ ]:
# ========== AUGMENTATION PIPELINE ==========
# Paper: Horizontal flip + Rotations (0, 30, 45, 90, 180, 270)
# We need to generate ~1468 total (4x original)
# 367 * 4 = 1468, so each original needs 3 augmented versions
# Paper uses 6 rotations + horizontal flip = 12 possible transforms
# We'll select a subset to get ~3 augmented per original

# Define the paper's rotation angles
PAPER_ROTATIONS = [0, 30, 45, 90, 180, 270]

# For 4x total, we need 3 augmented per original
# Strategy: For each original, apply 3 different augmentations:
# 1. Horizontal flip + rotation
# 2. Rotation only (different angle)
# 3. Rotation only (different angle) + pixel translation (our extension)

# Create augmentation transforms using Albumentations
# We'll create a transform for each rotation angle with/without flip

def get_augmentation_transforms():
    """Return list of augmentation transforms matching paper + pixel translation."""
    transforms = []
    
    # Base resize and normalize (applied to all)
    base_transform = A.Compose([
        A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    
    # Paper augmentations: horizontal flip + each rotation
    for angle in PAPER_ROTATIONS:
        # With horizontal flip
        transforms.append((f"flip_rot{angle}", A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Rotate(limit=(angle, angle), p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])))
        
        # Without horizontal flip (just rotation)
        transforms.append((f"rot{angle}", A.Compose([
            A.Rotate(limit=(angle, angle), p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])))
    
    # Our extension: pixel translation (Â±10-15%)
    # We'll add this as an additional transform option
    shift_limit = 0.15  # 15%
    for angle in [0, 45, 90, 180]:
        transforms.append((f"rot{angle}_translate", A.Compose([
            A.Rotate(limit=(angle, angle), p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.ShiftScaleRotate(shift_limit=shift_limit, scale_limit=0, rotate_limit=0, p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])))
    
    return transforms

all_transforms = get_augmentation_transforms()
print(f"Total available transforms: {len(all_transforms)}")
for name, _ in all_transforms:
    print(f"  {name}")

In [ ]:
# ========== GENERATE AUGMENTED DATASET ==========
# Strategy: For each class, generate augmented images to reach TARGET_PER_CLASS
# We'll cycle through transforms to distribute evenly

from tqdm import tqdm

# Create output directory structure
for class_name in EXPECTED_CLASSES:
    os.makedirs(os.path.join(OUTPUT_DIR, class_name), exist_ok=True)

# Group originals by class
originals_by_class = {}
for img_path, class_name in original_images:
    if class_name not in originals_by_class:
        originals_by_class[class_name] = []
    originals_by_class[class_name].append(img_path)

# Calculate how many augmented images needed per class
augmented_counts = {}
for class_name in EXPECTED_CLASSES:
    current = class_counts.get(class_name, 0)
    needed = max(0, TARGET_PER_CLASS - current)
    augmented_counts[class_name] = needed
    print(f"{class_name}: original={current}, target={TARGET_PER_CLASS}, need_augmented={needed}")

total_needed = sum(augmented_counts.values())
print(f"\nTotal augmented needed: {total_needed}")
print(f"Expected final total: {total_original + total_needed}")

In [ ]:
# ========== APPLY AUGMENTATIONS ==========
# We'll save augmented images as new files
# Use deterministic naming: original_name_augX.jpg

saved_count = {class_name: 0 for class_name in EXPECTED_CLASSES}
augmented_records = []

for class_name in EXPECTED_CLASSES:
    originals = originals_by_class.get(class_name, [])
    needed = augmented_counts[class_name]
    
    if needed == 0:
        # Just copy originals to output dir
        for img_path in originals:
            dst = Path(OUTPUT_DIR) / class_name / img_path.name
            shutil.copy2(img_path, dst)
            saved_count[class_name] += 1
            augmented_records.append({
                "path": str(dst),
                "class": class_name,
                "filename": img_path.name,
                "is_original": True,
                "transform": "original"
            })
        continue
    
    # Need to generate augmented images
    # First, copy originals
    for img_path in originals:
        dst = Path(OUTPUT_DIR) / class_name / img_path.name
        shutil.copy2(img_path, dst)
        saved_count[class_name] += 1
        augmented_records.append({
            "path": str(dst),
            "class": class_name,
            "filename": img_path.name,
            "is_original": True,
            "transform": "original"
        })
    
    # Generate augmented images
    # Cycle through transforms and originals
    transform_names = [name for name, _ in all_transforms]
    transform_idx = 0
    original_idx = 0
    
    with tqdm(total=needed, desc=f"Augmenting {class_name}") as pbar:
        while saved_count[class_name] < TARGET_PER_CLASS:
            img_path = originals[original_idx % len(originals)]
            transform_name, transform = all_transforms[transform_idx % len(all_transforms)]
            
            # Load and apply transform
            image = cv2.imread(str(img_path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            augmented = transform(image=image)
            aug_image = augmented["image"]
            
            # Denormalize for saving (convert from normalized float to uint8)
            # Albumentations Normalize outputs float32 in [0,1] range after normalization
            # We need to reverse: (normalized * std + mean) * 255
            mean = np.array(IMAGENET_MEAN).reshape(1, 1, 3)
            std = np.array(IMAGENET_STD).reshape(1, 1, 3)
            aug_image_denorm = (aug_image * std + mean) * 255
            aug_image_denorm = np.clip(aug_image_denorm, 0, 255).astype(np.uint8)
            
            # Save
            orig_stem = img_path.stem
            aug_filename = f"{orig_stem}_aug{transform_name}.jpg"
            dst = Path(OUTPUT_DIR) / class_name / aug_filename
            
            # Convert RGB back to BGR for cv2.imwrite
            aug_image_bgr = cv2.cvtColor(aug_image_denorm, cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(dst), aug_image_bgr)
            
            saved_count[class_name] += 1
            augmented_records.append({
                "path": str(dst),
                "class": class_name,
                "filename": aug_filename,
                "is_original": False,
                "transform": transform_name,
                "source_image": img_path.name
            })
            
            pbar.update(1)
            transform_idx += 1
            original_idx += 1

print(f"\nFinal counts per class:")
for class_name in EXPECTED_CLASSES:
    print(f"  {class_name}: {saved_count[class_name]}")
print(f"Total: {sum(saved_count.values())}")

In [ ]:
# ========== VERIFY AUGMENTED DATASET ==========
# Check saved images
verify_counts = {}
for class_name in EXPECTED_CLASSES:
    class_dir = Path(OUTPUT_DIR) / class_name
    img_files = list(class_dir.glob("*.jpg"))
    verify_counts[class_name] = len(img_files)
    print(f"{class_name}: {len(img_files)} images")

print(f"\nTotal verified: {sum(verify_counts.values())}")

# Quick visual check - sample a few augmented images
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, class_name in enumerate(EXPECTED_CLASSES):
    class_dir = Path(OUTPUT_DIR) / class_name
    aug_files = list(class_dir.glob("*_aug*.jpg"))
    if aug_files:
        sample = aug_files[0]
        img = cv2.imread(str(sample))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        row, col = divmod(i, 3)
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{class_name}\n{sample.name[:30]}...")
        axes[row, col].axis('off')

plt.suptitle('Sample Augmented Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'augmented_samples.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ========== SAVE COMBINED DATASET INFO ==========
# Save the augmented_records for use in split step
with open(os.path.join(DATA_INFO_DIR, 'combined_dataset_records.json'), 'w') as f:
    json.dump(augmented_records, f, indent=2)

# Also save as CSV
df_combined = pd.DataFrame(augmented_records)
df_combined.to_csv(os.path.join(DATA_INFO_DIR, 'combined_dataset_records.csv'), index=False)

# Save class distribution
dist_df = pd.DataFrame({
    'Class': list(verify_counts.keys()),
    'Count': list(verify_counts.values()),
    'Original_Count': [class_counts.get(c, 0) for c in EXPECTED_CLASSES],
    'Augmented_Count': [augmented_counts.get(c, 0) for c in EXPECTED_CLASSES]
})
dist_df.to_csv(os.path.join(DATA_INFO_DIR, 'combined_class_distribution.csv'), index=False)

# Save augmentation config
aug_config = {
    "seed": SEED,
    "target_total": TARGET_TOTAL,
    "target_per_class": TARGET_PER_CLASS,
    "img_size": IMG_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "paper_rotations": PAPER_ROTATIONS,
    "pixel_translation_limit": 0.15,
    "total_original": total_original,
    "total_augmented": total_needed,
    "final_total": sum(verify_counts.values()),
    "class_counts_original": class_counts,
    "class_counts_final": verify_counts,
    "transforms_used": [name for name, _ in all_transforms]
}

with open(os.path.join(DATA_INFO_DIR, 'augmentation_config.json'), 'w') as f:
    json.dump(aug_config, f, indent=2)

print(f"Combined dataset saved to: {OUTPUT_DIR}")
print(f"Records saved to: {DATA_INFO_DIR}/combined_dataset_records.json")
print(f"Config saved to: {DATA_INFO_DIR}/augmentation_config.json")


## Summary

- **Original images**: 367
- **Target augmented total**: 1,468 (4x)
- **Actual final total**: {{final_total}}
- **Classes**: 6 (balanced to ~244 each)
- **Augmentations**: Paper's horizontal flip + 6 rotations + pixel translation (Â±15%)
- **Output**: Combined dataset at `/kaggle/working/combined_dataset/`
- **Records**: `data_info/combined_dataset_records.json`

**Next**: Run `03_split.ipynb` (or proceed to split in next step) to create train/val/test splits.

---
*Note: Replace `{{final_total}}` with actual value from output above when updating memory.*